# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # metadata is an mlcroissant Metadata object
print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

Let's list out the available record sets, their fields, and columns using their `@id`. This information helps identify how the data is structured before loading records for further processing.

In [ ]:
# List all record sets and their @ids
record_sets = metadata.record_sets
if not record_sets:
    print('No record sets found in the metadata.')
else:
    print("Available Record Sets and their Fields:")
    for rs in record_sets:
        print(f"- Record Set Name: {rs.name}")
        print(f"  Record Set @id: {rs.id}")
        print("  Fields and Columns:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id} ; name: {field.name}")
            if hasattr(field, 'columns'):
                for column in field.columns:
                    print(f"      - Column @id: {column.id} ; name: {column.name}")
        print('')

## 3. Data Extraction
Load data from each available record set into DataFrames for analysis, referencing each by `@id`.

Below, we extract the records from all record sets into a dictionary of DataFrames, keyed by record set `@id`.

_If there are multiple record sets, this will create a DataFrame for each. The field and column names are drawn from the Croissant schema, using their `@id`._


In [ ]:
# Prepare the list of record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets] if metadata.record_sets else []

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"Columns for first record set ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print('No record sets with records found. Check Record Sets in previous cell.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on a numeric field, normalizing it, and grouping by a categorical field as referenced by their `@id`.

For demonstration, the notebook will dynamically select numeric and group fields if found. **If you know the exact field `@id`s to use, set them explicitly!**

In [ ]:
import numpy as np

if dataframes:
    # Use the first available record set
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to find a numeric field to use, otherwise prompt user
    numeric_field = None
    possible_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field for analysis: {numeric_field}")
    else:
        print("No numeric field found. Please edit the code to select a valid numeric field column by @id.")

    # Try to find a group/categorical field as well
    group_field = None
    possible_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if possible_group_fields:
        group_field = possible_group_fields[0]
        print(f"Using group field for grouping: {group_field}")

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0.0
        print(f"Filtering records with {numeric_field} > {threshold:.2f}")
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records (top 5):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
            / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records (top 5):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if available
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data (mean of {numeric_field} by {group_field}, top 5):")
            print(grouped_df.head())
    else:
        print("Please edit this cell with a valid numeric field column by the field's @id for EDA.")
else:
    print("No data available for EDA. Please ensure record sets have been loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field, and, if grouping is possible, the mean value by group.

_Feel free to adapt this for further visualization approaches!_

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if 'group_field' in locals() and group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: Data or field not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration:

- We successfully loaded and inspected the Croissant schema and explored available record sets and fields by their `@id`.
- Data from record sets was extracted into DataFrames, demonstrating how to use the `mlcroissant` library for structured data access.
- Basic EDA and normalization/grouping were performed dynamically based on the detected numeric and categorical fields.
- Simple data visualizations were generated for quick exploration.

Consider extending this notebook with domain-specific analyses relevant to the mass spectrometry proteomics dataset, such as statistical testing, biological annotation lookups, or interactive dashboards.

<!-- End of notebook -->